In [2]:
import regex as re
import importlib
import tokenizer
importlib.reload(tokenizer)
st = tokenizer.SimpleTokenizer
import data_loader
importlib.reload(data_loader)
gt  = data_loader.GPTDataset
from torch.utils.data import DataLoader

In [3]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of characters:",len(raw_text))
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
preprocessed_text = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed_text = [token.split()[0] for token in preprocessed_text if token.strip()]
print("Total number of tokens:",len(preprocessed_text))
print(preprocessed_text[:30])

Total number of tokens: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [5]:
ordered_list = sorted(set(preprocessed_text))
vocab_size = len(ordered_list)
print("Vocabulary size:",vocab_size)


Vocabulary size: 1130


In [6]:
vocab_dict = {token:idx for idx,token in enumerate(ordered_list)}
print("Vocabulary dictionary:",dict(list(vocab_dict.items())[:30]))

Vocabulary dictionary: {'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29}


In [7]:
t = st(raw_text)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = t.encode(text)
print("Encoded ids:",ids)

Encoded ids: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [8]:
decoded_text = t.decode(ids)
print("Decoded text:",decoded_text)

Decoded text: " It's the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [9]:
text = "Hello, world! It' s a test."
ids = t.encode(text)
print("Encoded ids:",ids)

Encoded ids: [1134, 5, 1134, 0, 56, 2, 850, 115, 1134, 7]


In [10]:
text1 = "Hello, world! It's a test."
text2 = "Here is such a big palace that it can be used to store a lot of things."
text = "<|BOS|> " + text1 + " <|EOS|> " + text2 + " <|endoftext|>"
ids = t.encode(text)
print("Encoded ids:",ids)

Encoded ids: [1131, 1134, 5, 1134, 0, 56, 2, 850, 115, 1134, 7, 1133, 1134, 584, 949, 115, 219, 1134, 987, 585, 244, 198, 1057, 1016, 1134, 115, 1134, 722, 997, 7, 1130]


In [11]:
import tiktoken
print("tiktoken version:", importlib.metadata.version("tiktoken"))


tiktoken version: 0.13.0


In [12]:
tokenizer = tiktoken.get_encoding("gpt2")

In [13]:
text1 = "Hello, world! It's a test."
text2 = "Here is such a big palace that it can be used to store alotofthings."
text = " <|endoftext|> ".join([text1, text2])
ids = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print("Encoded ids:",ids)

Encoded ids: [15496, 11, 995, 0, 632, 338, 257, 1332, 13, 220, 50256, 3423, 318, 884, 257, 1263, 20562, 326, 340, 460, 307, 973, 284, 3650, 43158, 1659, 27971, 13]


In [14]:
text_decoded = tokenizer.decode(ids)
print("Decoded text:",text_decoded)

Decoded text: Hello, world! It's a test. <|endoftext|> Here is such a big palace that it can be used to store alotofthings.


In [15]:
enc_text = tokenizer.encode(raw_text)
len_enc_text = len(enc_text)
print("Length of encoded text:",len_enc_text)

Length of encoded text: 5145


In [16]:
enc_sample = enc_text[50:]

In [17]:
context_size = 4
input_array = enc_sample[:context_size]
output_array = enc_sample[1:context_size+1]
print("Input array:",input_array)
print("Output array:",output_array)

Input array: [290, 4920, 2241, 287]
Output array: [4920, 2241, 287, 257]


In [18]:
for i in range(context_size):
    context = input_array[:i+1]
    output = output_array[i]
    print(f"Context: {context}, Output: {output}")

Context: [290], Output: 4920
Context: [290, 4920], Output: 2241
Context: [290, 4920, 2241], Output: 287
Context: [290, 4920, 2241, 287], Output: 257


In [19]:
for i in range(context_size):
    context = enc_sample[:i+1]
    output = enc_sample[i+1]
    print(f"Context: {tokenizer.decode(context)}, Output: {tokenizer.decode([output])}")

Context:  and, Output:  established
Context:  and established, Output:  himself
Context:  and established himself, Output:  in
Context:  and established himself in, Output:  a


In [20]:
def create_data_loader(txt, batch_size=4, context_size=256,
                       stride=128,shuffle=True,drop_last=True,
                       cpu_thread_number=4):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = gt(txt, tokenizer, context_size, stride)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                             drop_last=drop_last, num_workers=cpu_thread_number)
    return data_loader


In [21]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


In [22]:
dataloader = create_data_loader(raw_text, batch_size=10, context_size=4, stride=1,
                                shuffle=False, drop_last=True, cpu_thread_number=4)
data_iter = iter(dataloader)
first_batch = next(data_iter)

In [23]:
print("First batch Input:", first_batch[0])
print("First batch Target:", first_batch[1])

First batch Input: tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026],
        [ 2138,   257,  7026, 15632]])
First batch Target: tensor([[  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026],
        [ 2138,   257,  7026, 15632],
        [  257,  7026, 15632,   438]])


In [25]:
import gensim.downloader as api
word_vectors = api.load("word2vec-google-news-300")

In [26]:
print(word_vectors['king'])

[ 1.25976562e-01  2.97851562e-02  8.60595703e-03  1.39648438e-01
 -2.56347656e-02 -3.61328125e-02  1.11816406e-01 -1.98242188e-01
  5.12695312e-02  3.63281250e-01 -2.42187500e-01 -3.02734375e-01
 -1.77734375e-01 -2.49023438e-02 -1.67968750e-01 -1.69921875e-01
  3.46679688e-02  5.21850586e-03  4.63867188e-02  1.28906250e-01
  1.36718750e-01  1.12792969e-01  5.95703125e-02  1.36718750e-01
  1.01074219e-01 -1.76757812e-01 -2.51953125e-01  5.98144531e-02
  3.41796875e-01 -3.11279297e-02  1.04492188e-01  6.17675781e-02
  1.24511719e-01  4.00390625e-01 -3.22265625e-01  8.39843750e-02
  3.90625000e-02  5.85937500e-03  7.03125000e-02  1.72851562e-01
  1.38671875e-01 -2.31445312e-01  2.83203125e-01  1.42578125e-01
  3.41796875e-01 -2.39257812e-02 -1.09863281e-01  3.32031250e-02
 -5.46875000e-02  1.53198242e-02 -1.62109375e-01  1.58203125e-01
 -2.59765625e-01  2.01416016e-02 -1.63085938e-01  1.35803223e-03
 -1.44531250e-01 -5.68847656e-02  4.29687500e-02 -2.46582031e-02
  1.85546875e-01  4.47265

In [27]:
print(word_vectors['queen'].shape)

(300,)


In [28]:
print(word_vectors.similarity('king', 'queen'))

0.6510957


In [29]:
print(word_vectors.most_similar(positive=['king','woman'], negative=['man'], topn=10))

[('queen', 0.7118193507194519), ('monarch', 0.6189674139022827), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321839332581), ('kings', 0.5236844420433044), ('Queen_Consort', 0.5235945582389832), ('queens', 0.518113374710083), ('sultan', 0.5098593235015869), ('monarchy', 0.5087411403656006)]


In [36]:
print(word_vectors.similarity('apple', 'man'))
print(word_vectors.similarity('apple', 'orange'))
print(word_vectors.similarity('apple', 'pear'))
print(word_vectors.similarity('apple', 'heaven'))
print(word_vectors.similarity('apple', 'Adam'))
print(word_vectors.similarity('apple', 'Eve'))
print(word_vectors.similarity('apple', 'Satan'))

0.11685415
0.39203462
0.64506966
0.11087046
0.13577957
0.25482452
0.13343525


In [37]:
input_ids = torch.tensor([2,3,5,1])

In [38]:
vocab_size = 6
vec_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, vec_dim)

In [39]:
print("Embedding layer weights:", embedding_layer.weight)

Embedding layer weights: Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [40]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [41]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [42]:
vocab_size = 50257
vec_dim = 256
embedding_layer = torch.nn.Embedding(vocab_size, vec_dim)

In [47]:
context_length = 4
dataloader = create_data_loader(raw_text, batch_size=8, context_size=context_length, stride=context_length,
                                shuffle=False)
data_iter = iter(dataloader)
first_batch_input,first_batch_target = next(data_iter)

In [48]:
print("Input Token IDs:\t",first_batch_input)
print("Input Shape:\t",first_batch_input.shape)

Input Token IDs:	 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Input Shape:	 torch.Size([8, 4])


In [49]:
token_embedding = embedding_layer(first_batch_input)
print("Token Embedding Shape:\t",token_embedding.shape)

Token Embedding Shape:	 torch.Size([8, 4, 256])


In [50]:
pos_embedding_layer = torch.nn.Embedding(context_length, vec_dim)
print("Positional Embedding Layer Size:\t",pos_embedding_layer.weight.shape)

Positional Embedding Layer Size:	 torch.Size([4, 256])


In [51]:
pos_embedding_vectors = pos_embedding_layer(torch.arange(context_length))
print("Positional Embedding Vectors Shape:\t",pos_embedding_vectors.shape)

Positional Embedding Vectors Shape:	 torch.Size([4, 256])


In [52]:
first_batch_input_with_pos = token_embedding + pos_embedding_vectors
print("First Batch Input with Positional Embedding Shape:\t",first_batch_input_with_pos.shape)

First Batch Input with Positional Embedding Shape:	 torch.Size([8, 4, 256])
